## 4.1 ディープラーニング向けライブラリの導入

### 4.1.1 Keras

In [11]:
import numpy as np
import tensorflow as tf
from sklearn import datasets
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import optimizers

In [ ]:
np.random.seed(123)
tf.random.set_seed(123)  # tensorflow用の乱数シード
"""
1. データの準備
"""
N = 300
x, t = datasets.make_moons(N, noise=0.3)
t = t.reshape(N, 1)
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.2)

"""
2. モデルの構築
"""
# Kerasではモデル構築を「ネットワークに層を追加していく」ことで直観的に記述できる.
# Sequentialクラスでネットワークをセットアップ.
model = Sequential()
# 空のモデルにaddメソッドで層を追加していく. chapter3のLayerがDenseに相当.
model.add(Dense(3, activation="sigmoid"))  # 隠れ層を追加
model.add(Dense(1, activation="sigmoid"))  # 出力層を追加

"""
3. モデルの学習
"""
optimizer = optimizers.SGD(learning_rate=0.1)
# モデルをどのように学習するかをcompileメソッドで設定.
model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
# 学習を行う.
model.fit(x_train, t_train, epochs=100, batch_size=10, verbose=1)

"""
4. モデルの評価
"""
loss, acc = model.evaluate(x_test, t_test, verbose=0)
display(f"test_loss: {loss:.3f}, test_acc: {acc:.3f}")
# 出力はmodel.predict(x_test)で得られる. 分類後出力はmodel.predict_classes(x_test)で得られる.

Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5393 - loss: 0.6884  
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5393 - loss: 0.6770 
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5698 - loss: 0.6651 
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6699 - loss: 0.6525 
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7649 - loss: 0.6392 
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7932 - loss: 0.6251 
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7903 - loss: 0.6105 
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7998 - loss: 0.5955 
Epoch 9/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8098 - loss: 0.5805 
Epoch 10/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7976 - loss: 0.5657 
Epoch 11/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7963 - loss: 0.5515 
Epoch 12/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

'test_loss: 0.306, test_acc: 0.867'

### 4.1.2 TensorFlow

In [12]:
from typing import Any

import numpy as np
import tensorflow as tf
from sklearn import datasets
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from tensorflow.keras import optimizers
from tensorflow.keras import losses
from tensorflow.keras import metrics

In [ ]:
np.random.seed(123)
tf.random.set_seed(123)
"""
1. データの準備
"""
N = 300
x, t = datasets.make_moons(N, noise=0.3)
t = t.reshape(N, 1)
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.2)

"""
2. モデルの構築
"""


# tensorflow.keras.Modelをカスタマイズする形で実装.
class MLP(Model):
    """
    多層パーセプトロン
    """

    def __init__(self, hidden_dim: int, output_dim: int) -> None:
        super().__init__()
        # Layerの代わりにDenseを使う.
        self.l1 = Dense(hidden_dim, activation="sigmoid")
        self.l2 = Dense(output_dim, activation="sigmoid")

    def call(self, x: np.ndarray) -> np.ndarray:
        # 順伝播計算.
        h = self.l1(x)
        y = self.l2(h)
        return y


model = MLP(3, 1)

"""
3. モデルの学習
"""
criterion = losses.BinaryCrossentropy()
optimizer = optimizers.SGD(learning_rate=0.1)


def compute_loss(t: np.ndarray, y: np.ndarray) -> tf.Tensor:
    return criterion(t, y)


def train_step(model: MLP, x: np.ndarray, t: np.ndarray) -> tuple[tf.Tensor, MLP]:
    # tensorflowが自動で勾配を計算.
    with tf.GradientTape() as tape:
        preds = model(x)
        loss = compute_loss(t, preds)
    grads = tape.gradient(loss, model.trainable_variables)
    # 勾配からパラメータを更新.
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss, model


epochs = 100
batch_size = 10
n_batches = x_train.shape[0] // batch_size
for epoch in range(epochs):
    train_loss = 0.0
    x_, t_ = shuffle(x_train, t_train)
    for batch in range(n_batches):
        start = batch * batch_size
        end = start + batch_size
        loss, model = train_step(model, x_[start:end], t_[start:end])
        train_loss += loss.numpy()
    display(f"epoch: {epoch + 1}, loss: {train_loss:.3f}")

"""
4. モデルの評価
"""
test_loss = metrics.Mean()  # 誤差関数を求めるためのオブジェクト
test_acc = metrics.BinaryAccuracy()  # 正解率を求めるためのオブジェクト


def test_step(model: MLP, x: np.ndarray, t: np.ndarray) -> tuple[np.ndarray, Any, Any]:
    preds = model(x)
    loss = compute_loss(t, preds)
    test_loss(loss)
    test_acc(t, preds)
    return preds, test_loss, test_acc


_, test_loss, test_acc = test_step(model, x_test, t_test)
display(f"test_loss: {test_loss.result():.3f}, test_acc: {test_acc.result():.3f}")

'epoch: 1, loss: 16.484'

'epoch: 2, loss: 15.870'

'epoch: 3, loss: 15.347'

'epoch: 4, loss: 14.840'

'epoch: 5, loss: 14.379'

...

'epoch: 97, loss: 9.324'

'epoch: 98, loss: 9.323'

'epoch: 99, loss: 9.323'

'epoch: 100, loss: 9.320'

'test_loss: 0.313, test_acc: 0.867'

### 4.1.3 PyTorch

In [13]:
from typing import Callable

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optimizers

from sklearn import datasets
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.nn.modules import Module
from torch.optim import Optimizer

In [ ]:
class MLP(Module):
    """
    多層パーセプトロン
    """

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int) -> None:
        super().__init__()
        # KerasのDenseに対応する.
        self.l1 = nn.Linear(input_dim, hidden_dim)
        # 活性化関数は別で定義.
        self.a1 = nn.Sigmoid()
        self.l2 = nn.Linear(hidden_dim, output_dim)
        self.a2 = nn.Sigmoid()
        # 順伝播計算用のリスト
        self.layers = [self.l1, self.a2, self.l2, self.a2]

    def forward(self, x: np.ndarray) -> np.ndarray:
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
np.random.seed(123)
torch.manual_seed(123)  # torch用の乱数シード
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
"""
1. データの準備
"""
N = 300
x, t = datasets.make_moons(N, noise=0.3)
t = t.reshape(N, 1)
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.2)

"""
2. モデルの構築
"""
# GPU利用可能な環境ならGPUを利用するように設定.
model = MLP(2, 3, 1).to(device)

"""
3. モデルの学習
"""
criterion = nn.BCELoss()
optimizer = optimizers.SGD(model.parameters(), lr=0.1)


def compute_loss(t: torch.Tensor, y: torch.Tensor) -> Module:
    return criterion(y, t)


def train_step(
    model: Module, optimizer: Optimizer, x: torch.Tensor, t: torch.Tensor
) -> tuple[Module, Module, Optimizer]:
    # モデルを学習モードに切り替える.
    model.train()
    preds = model(x)
    loss = compute_loss(t, preds)
    optimizer.zero_grad()  # 勾配の初期化
    loss.backward()  # 勾配計算
    optimizer.step()  # パラメータ更新
    return loss, model, optimizer


epochs = 100
batch_size = 10
n_batches = x_train.shape[0] // batch_size
for epoch in range(epochs):
    train_loss = 0.0
    x_, t_ = shuffle(x_train, t_train)
    # numpy配列データをtorch.Tensorに変換.
    x_ = torch.Tensor(x_).to(device)
    t_ = torch.Tensor(t_).to(device)
    for n_batch in range(n_batches):
        start = n_batch * batch_size
        end = start + batch_size
        loss, model, optimizer = train_step(
            model, optimizer, x_[start:end], t_[start:end]
        )
        train_loss += loss.item()
    display(f"epoch: {epoch + 1}, loss: {train_loss:.3f}")

"""
4. モデルの評価
"""


def test_step(
    model: Module, x: np.ndarray, t: np.ndarray
) -> tuple[Module, torch.Tensor]:
    x = torch.Tensor(x).to(device)
    t = torch.Tensor(t).to(device)
    # モデルを推論モードに切り替える.
    model.eval()
    preds = model(x)
    loss = compute_loss(t, preds)
    return loss, preds


loss, preds = test_step(model, x_test, t_test)
test_loss = loss.item()
# モデルの出力をnumpy配列に変換
# preds: 勾配計算の情報を含むテンソル型
# preds.data: 勾配計算の情報を含まないテンソル型
# preds.data.cpu(): CPUのテンソル型
# preds.data.cpu().numpy(): numpy配列
preds = preds.data.cpu().numpy() > 0.5
test_acc = accuracy_score(t_test, preds)
display(f"test_loss: {test_loss:.3f}, test_acc: {test_acc:.3f}")

'epoch: 1, loss: 17.123'

'epoch: 2, loss: 16.806'

'epoch: 3, loss: 16.587'

'epoch: 4, loss: 16.362'

'epoch: 5, loss: 16.137'

...

'epoch: 97, loss: 9.370'

'epoch: 98, loss: 9.367'

'epoch: 99, loss: 9.368'

'epoch: 100, loss: 9.364'

'test_loss: 0.302, test_acc: 0.867'

## 4.2 ディープラーニングへの準備

### 4.2.1 学習における問題

In [14]:
from tensorflow.keras import datasets


mnist = datasets.mnist
(x_train, t_train), (x_test, t_test) = mnist.load_data()

In [28]:
x_train = (x_train.reshape(-1, 784) / 255).astype(np.float32)
x_test = (x_test.reshape(-1, 784) / 255).astype(np.float32)
t_train = np.eye(10)[t_train].astype(np.float32)  # 1-of-K表現に変換
t_test = np.eye(10)[t_test].astype(np.float32)

In [31]:
model = Sequential()
model.add(Dense(200, activation="sigmoid"))
model.add(Dense(10, activation="softmax"))
model.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train, t_train, epochs=30, batch_size=100, verbose=2)

loss, acc = model.evaluate(x_test, t_test, verbose=0)
display(f"test_loss: {loss:.3f}, test_acc: {acc:.3f}")

Epoch 1/30
600/600 - 1s - 2ms/step - accuracy: 0.5541 - loss: 1.9457
Epoch 2/30
600/600 - 1s - 2ms/step - accuracy: 0.7692 - loss: 1.3301
Epoch 3/30
600/600 - 1s - 2ms/step - accuracy: 0.8164 - loss: 0.9715
Epoch 4/30
600/600 - 1s - 2ms/step - accuracy: 0.8392 - loss: 0.7802
Epoch 5/30
600/600 - 1s - 2ms/step - accuracy: 0.8518 - loss: 0.6686
Epoch 6/30
600/600 - 1s - 2ms/step - accuracy: 0.8616 - loss: 0.5967
Epoch 7/30
600/600 - 1s - 2ms/step - accuracy: 0.8691 - loss: 0.5466
Epoch 8/30
600/600 - 1s - 2ms/step - accuracy: 0.8745 - loss: 0.5097
Epoch 9/30
600/600 - 1s - 2ms/step - accuracy: 0.8787 - loss: 0.4814
Epoch 10/30
600/600 - 1s - 2ms/step - accuracy: 0.8819 - loss: 0.4590
Epoch 11/30
600/600 - 1s - 2ms/step - accuracy: 0.8853 - loss: 0.4410
Epoch 12/30
600/600 - 1s - 2ms/step - accuracy: 0.8880 - loss: 0.4259
Epoch 13/30
600/600 - 1s - 2ms/step - accuracy: 0.8901 - loss: 0.4132
Epoch 14/30
600/600 - 1s - 2ms/step - accuracy: 0.8918 - loss: 0.4024
Epoch 15/30
600/600 - 1s - 2m

'test_loss: 0.308, test_acc: 0.914'

In [32]:
model = Sequential()
model.add(Dense(200, activation="sigmoid"))
model.add(Dense(200, activation="sigmoid"))
model.add(Dense(200, activation="sigmoid"))
model.add(Dense(10, activation="softmax"))
model.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train, t_train, epochs=30, batch_size=100, verbose=2)

loss, acc = model.evaluate(x_test, t_test, verbose=0)
display(f"test_loss: {loss:.3f}, test_acc: {acc:.3f}")

Epoch 1/30
600/600 - 2s - 3ms/step - accuracy: 0.1103 - loss: 2.3046
Epoch 2/30
600/600 - 1s - 2ms/step - accuracy: 0.1231 - loss: 2.2957
Epoch 3/30
600/600 - 1s - 2ms/step - accuracy: 0.1337 - loss: 2.2903
Epoch 4/30
600/600 - 1s - 2ms/step - accuracy: 0.1601 - loss: 2.2839
Epoch 5/30
600/600 - 1s - 2ms/step - accuracy: 0.1771 - loss: 2.2763
Epoch 6/30
600/600 - 1s - 2ms/step - accuracy: 0.2128 - loss: 2.2662
Epoch 7/30
600/600 - 1s - 2ms/step - accuracy: 0.2647 - loss: 2.2520
Epoch 8/30
600/600 - 1s - 2ms/step - accuracy: 0.3070 - loss: 2.2311
Epoch 9/30
600/600 - 1s - 2ms/step - accuracy: 0.3399 - loss: 2.1982
Epoch 10/30
600/600 - 1s - 2ms/step - accuracy: 0.3670 - loss: 2.1433
Epoch 11/30
600/600 - 1s - 2ms/step - accuracy: 0.3883 - loss: 2.0512
Epoch 12/30
600/600 - 1s - 2ms/step - accuracy: 0.4241 - loss: 1.9134
Epoch 13/30
600/600 - 1s - 2ms/step - accuracy: 0.4732 - loss: 1.7478
Epoch 14/30
600/600 - 1s - 2ms/step - accuracy: 0.5134 - loss: 1.5827
Epoch 15/30
600/600 - 1s - 2m

'test_loss: 0.587, test_acc: 0.829'

### 4.2.4　実装の準備

#### 4.2.4.1 Keras

省略(4.2.1参照)

#### 4.2.4.2 TensorFlow

In [43]:
from typing import Any

import numpy as np
import tensorflow as tf

from sklearn.utils import shuffle
from tensorflow.keras import datasets
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from tensorflow.keras import optimizers
from tensorflow.keras import losses
from tensorflow.keras import metrics

In [44]:
class DNN(Model):
    def __init__(self, hidden_dim: int, output_dim: int) -> None:
        super().__init__()
        self.l1 = Dense(hidden_dim, activation="sigmoid")
        self.l2 = Dense(hidden_dim, activation="sigmoid")
        self.l3 = Dense(hidden_dim, activation="sigmoid")
        self.l4 = Dense(output_dim, activation="softmax")
        self.ls = [self.l1, self.l2, self.l3, self.l4]

    def call(self, x: np.ndarray) -> np.ndarray:
        for layer in self.ls:
            x = layer(x)
        return x

In [ ]:
# Kerasより遅く感じる
np.random.seed(123)
tf.random.set_seed(123)
"""
1. データの準備
"""
mnist = datasets.mnist
(x_train, t_train), (x_test, t_test) = mnist.load_data()

x_train = (x_train.reshape(-1, 784) / 255).astype(np.float32)
x_test = (x_test.reshape(-1, 784) / 255).astype(np.float32)
t_train = np.eye(10)[t_train].astype(np.float32)
t_test = np.eye(10)[t_test].astype(np.float32)

"""
2. モデルの構築
"""
model = DNN(200, 10)

"""
3. モデルの学習
"""
criterion = losses.CategoricalCrossentropy()  # 多クラス問題の分類損失関数
optimizer = optimizers.SGD(learning_rate=0.01)
train_loss = metrics.Mean()
train_acc = metrics.CategoricalAccuracy()  # 多クラス問題の正解率


def compute_loss(t: np.ndarray, y: np.ndarray) -> tf.Tensor:
    return criterion(t, y)


def train_step(
    model: DNN,
    optimizer: Any,
    trian_loss: Any,
    tran_acc: Any,
    x: np.ndarray,
    t: np.ndarray,
) -> tuple[tf.Tensor, DNN, Any, Any, Any]:
    with tf.GradientTape() as tape:
        preds = model(x)
        loss = compute_loss(t, preds)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    train_loss(loss)
    train_acc(t, preds)
    return loss, model, optimizer, train_loss, train_acc


epochs = 30
batch_size = 100
n_batches = x_train.shape[0] // batch_size
for epoch in range(epochs):
    x_, t_ = shuffle(x_train, t_train)
    for batch in range(n_batches):
        start = batch * batch_size
        end = start + batch_size
        _, model, optimizer, train_loss, train_acc = train_step(
            model, optimizer, train_loss, train_acc, x_[start:end], t_[start:end]
        )
    display(
        f"epoch: {epoch + 1}, loss: {train_loss.result():.3f}, acc: {train_acc.result():.3f}"
    )

"""
4. モデルの評価
"""
test_loss = metrics.Mean()
test_acc = metrics.CategoricalAccuracy()


def test_step(
    test_loss: Any, test_acc: Any, x: np.ndarray, t: np.ndarray
) -> tuple[tf.Tensor, Any, Any]:
    preds = model(x)
    loss = compute_loss(t, preds)
    test_loss(loss)
    test_acc(t, preds)
    return loss, test_loss, test_acc


_, test_loss, test_acc = test_step(test_loss, test_acc, x_test, t_test)
display(f"test_loss: {test_loss.result():.3f}, test_acc: {test_acc.result():.3f}")

'epoch: 1, loss: 2.302, acc: 0.122'

'epoch: 2, loss: 2.298, acc: 0.130'

'epoch: 3, loss: 2.295, acc: 0.137'

'epoch: 4, loss: 2.291, acc: 0.142'

'epoch: 5, loss: 2.288, acc: 0.151'

...

'epoch: 27, loss: 1.574, acc: 0.495'

'epoch: 28, loss: 1.541, acc: 0.506'

'epoch: 29, loss: 1.510, acc: 0.517'

'epoch: 30, loss: 1.480, acc: 0.527'

'test_loss: 0.586, test_acc: 0.830'

#### 4.2.4.3 PyTorch

In [40]:
import os
from torchvision import datasets


root = os.path.join("~", ".torch", "mnist")
mnist_train = datasets.MNIST(root=root, download=True, train=True)
for datum in mnist_train:
    display(datum)
    break

100.0%
100.0%
100.0%
100.0%


(<PIL.Image.Image image mode=L size=28x28>, 5)

In [46]:
from torchvision import datasets
import torchvision.transforms as transforms


# transforms.ToTensor: Imageオブジェクトをテンソル型に変換
# lambda x: x.view(-1): テンソルの次元を(28, 28)から(784,)に変換
transform = transforms.Compose([transforms.ToTensor(), lambda x: x.view(-1)])
mnist_train = datasets.MNIST(root=root, download=True, train=True, transform=transform)

In [ ]:
from torch.utils.data import DataLoader


# batch_size: バッチ単位でデータをイテレーションできる
# shuffle: 各エポックでデータシャッフル処理を行ってくれる
train_dataloader = DataLoader(mnist_train, batch_size=100, shuffle=True)

In [48]:
for (x, t) in train_dataloader:
    display(x.shape)
    break

torch.Size([100, 784])

In [52]:
import os

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optimizers
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader
from torchvision import datasets
from torch.optim import Optimizer

In [51]:
class DNN(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int) -> None:
        super().__init__()
        self.l1 = nn.Linear(input_dim, hidden_dim)
        self.a1 = nn.Sigmoid()
        self.l2 = nn.Linear(hidden_dim, hidden_dim)
        self.a2 = nn.Sigmoid()
        self.l3 = nn.Linear(hidden_dim, hidden_dim)
        self.a3 = nn.Sigmoid()
        self.l4 = nn.Linear(hidden_dim, output_dim)
        self.layers = [self.l1, self.a1, self.l2, self.a2, self.l3, self.a3, self.l4]

    def forward(self, x: np.ndarray) -> np.ndarray:
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
# 正解率が低いのはなぜ？
np.random.seed(123)
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
"""
1. データの準備
"""
root = os.path.join("~", ".torch", "mnist")
transform = transforms.Compose([transforms.ToTensor(), lambda x: x.view(-1)])
mnist_train = datasets.MNIST(root=root, download=True, train=True, transform=transform)
mnist_test = datasets.MNIST(root=root, download=True, train=False, transform=transform)
train_dataloader = DataLoader(mnist_train, batch_size=100, shuffle=True)
test_dataloader = DataLoader(mnist_test, batch_size=100, shuffle=False)

"""
2. モデルの構築
"""
model = DNN(784, 200, 10).to(device)

"""
3. モデルの学習
"""
criterion = nn.CrossEntropyLoss()
optimizer = optimizers.SGD(model.parameters(), lr=0.01)


def compute_loss(t: torch.Tensor, y: torch.Tensor) -> nn.Module:
    return criterion(y, t)


def train_step(
    model: nn.Module, optimizer: Optimizer, x: torch.Tensor, t: torch.Tensor
) -> tuple[nn.Module, nn.Module, nn.Module, Optimizer]:
    model.train()
    preds = model(x)
    loss = compute_loss(t, preds)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss, preds, model, optimizer


epochs = 30
for epoch in range(epochs):
    train_loss = 0.0
    train_acc = 0.0
    for x, t in train_dataloader:
        x, t = x.to(device), t.to(device)
        loss, preds, model, optimizer = train_step(model, optimizer, x, t)
        train_loss += loss.item()
        train_acc += accuracy_score(t.tolist(), preds.argmax(dim=-1).tolist())
    train_loss /= len(train_dataloader)
    train_acc /= len(train_dataloader)
    display(f"epoch: {epoch + 1}, loss: {train_loss:.3f}, acc: {train_acc:.3f}")

"""
4. モデルの評価
"""


def test_step(
    model: nn.Module, x: torch.Tensor, t: torch.Tensor
) -> tuple[nn.Module, torch.Tensor, nn.Module]:
    model.eval()
    preds = model(x)
    loss = criterion(preds, t)
    return loss, preds, model


test_loss = 0.0
test_acc = 0.0
for x, t in test_dataloader:
    x, t = x.to(device), t.to(device)
    loss, preds, model = test_step(model, x, t)
    test_loss += loss.item()
    test_acc += accuracy_score(t.tolist(), preds.argmax(dim=-1).tolist())
test_loss /= len(test_dataloader)
test_acc /= len(test_dataloader)
display(f"test_loss: {test_loss:.3f}, test_acc: {test_acc:.3f}")

'epoch: 1, loss: 2.303, acc: 0.111'

'epoch: 2, loss: 2.302, acc: 0.111'

'epoch: 3, loss: 2.302, acc: 0.110'

'epoch: 4, loss: 2.302, acc: 0.109'

'epoch: 5, loss: 2.302, acc: 0.112'

...

'epoch: 27, loss: 2.297, acc: 0.121'

'epoch: 28, loss: 2.297, acc: 0.122'

'epoch: 29, loss: 2.296, acc: 0.124'

'epoch: 30, loss: 2.295, acc: 0.128'

'test_loss: 2.294, test_acc: 0.113'